In [1]:
import numpy as np, scipy
import librosa, librosa.display
import sklearn
import os
import warnings
import tqdm
import json
import pickle
import pandas as pd

warnings.filterwarnings('ignore')

In [2]:
# Retrieve the target based on the directories
SOUND_FOLDER = "./Audio_Dataset"
FEAT_FOLDER = "./Features"

In [3]:
# Utilities
def normalize(x, axis=0):
    return sklearn.preprocessing.minmax_scale(x, axis=axis)

def sigmoid_function(x):
    return 1.0/(1.0+np.exp(-x))


---
# **FEATURE EXTRACTION**




In [4]:
# Feature extraction
def extract_spectral_centroid(y,sr):
    spectral_centroid = librosa.feature.spectral_centroid(y=y+0.01, sr=sr)[0]
    return [spectral_centroid, float("{:.2f}".format(np.mean(spectral_centroid)))]

def extract_spectral_band_width(y,sr):
    spectral_band_width = librosa.feature.spectral_bandwidth(y=y+0.01, sr=sr, p=3)[0]
    return [spectral_band_width, float("{:.2f}".format(np.mean(spectral_band_width)))]

def extract_mfcc(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    return mfcc, mfcc_mean

def extract_spectral_contrast(y, sr):
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)
    return contrast, contrast_mean


In [5]:
path = os.path.join(SOUND_FOLDER)

In [6]:
# Retrieve list of files
list_of_files = []

# dirs=directories
for (root, dirs, file) in os.walk(path):
    for f in file:
        if '.wav' in f:
            list_of_files.append(root+"/"+f)

In [7]:
# Define the Dataframes of features
feat_df = pd.DataFrame(columns=['file_name'])

In [8]:
# -- FEATURE EXTRECTION --
# Create the dictionary list
#feat_dict = list()
#d = dict()

# Define the Dataframes of features
#feat_df = pd.DataFrame(columns=['file_name'])

for fname in tqdm.tqdm(list_of_files):
  #d['path'] = fname
  #d['file_name'] = os.path.basename(fname).split('/')[-1]
  signal, sr = librosa.load(path=fname)

  db_new_row = {'path': fname}
  db_new_row['file_name'] = os.path.basename(fname).split('/')[-1]

  # Spectral Centroid
  feat, feat_mean = extract_spectral_centroid(y=signal, sr=sr)
  #d['SC_mean'] = feat_mean
  db_new_row['SC_mean'] = feat_mean

  # Spectral BandWidth
  feat, feat_mean = extract_spectral_band_width(y=signal, sr=sr)
  #d['SBW_mean'] = feat_mean
  db_new_row['SBW_mean'] = feat_mean
  
  # MFCC
  feat, feat_mean = extract_mfcc(y=signal, sr=sr)
  #d['MFCC_mean'] = feat_mean.tolist()
  for i, val in enumerate(feat_mean):
    db_new_row["MFCC_mean_"+str(i)] = val

  # Spectral Contrast
  feat, feat_mean = extract_spectral_contrast(signal,sr)
  #d['Contrast_mean'] = feat_mean.tolist()
  for i, val in enumerate(feat_mean):
    db_new_row["Contrast_mean"+str(i)] = val

  #feat_dict.append(d.copy())
  s = pd.Series(db_new_row)
  feat_df = pd.concat([feat_df, s.to_frame().T], ignore_index=True)

  0%|          | 0/160 [00:00<?, ?it/s]

100%|██████████| 160/160 [00:11<00:00, 13.65it/s]


In [9]:
feat_df

,file_name,path,SC_mean,SBW_mean,MFCC_mean_0,MFCC_mean_1,MFCC_mean_2,MFCC_mean_3,MFCC_mean_4,MFCC_mean_5,...,MFCC_mean_10,MFCC_mean_11,MFCC_mean_12,Contrast_mean0,Contrast_mean1,Contrast_mean2,Contrast_mean3,Contrast_mean4,Contrast_mean5,Contrast_mean6
0,Bass Sample 40.wav,./Audio_Dataset/kick/Bass Sample 40.wav,150.75,1015.73,-532.827271,31.943409,17.710817,12.833717,9.849709,9.16721,...,3.3579,3.218079,3.006,21.344144,14.500125,13.399613,15.123102,19.392441,19.998107,29.596952
1,Bass Sample 8.wav,./Audio_Dataset/kick/Bass Sample 8.wav,417.12,2382.5,-493.95929,35.429024,22.820415,27.602272,22.493082,20.092983,...,13.518474,11.085154,9.577058,15.871044,13.586056,14.40261,13.255993,14.459496,14.98562,42.520382
2,Bass Sample 23.wav,./Audio_Dataset/kick/Bass Sample 23.wav,197.32,1448.13,-537.407654,29.824848,18.296659,21.677267,16.876102,14.519204,...,5.045534,3.249687,2.544477,14.90903,17.027073,14.299423,14.140297,15.249439,16.002135,39.344684
3,Bass Sample 37.wav,./Audio_Dataset/kick/Bass Sample 37.wav,371.22,2010.63,-449.292969,26.058981,13.255219,18.491783,11.475267,12.965648,...,7.157711,7.861504,4.533994,20.283957,13.854929,13.226863,13.593977,15.003669,15.396414,44.672291
4,Bass Sample 36.wav,./Audio_Dataset/kick/Bass Sample 36.wav,326.61,1836.81,-462.241547,23.550018,11.211489,17.141399,9.932425,11.08899,...,7.373727,9.010051,5.0444,17.982518,13.352796,13.688051,13.82182,15.317164,16.485585,43.568185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Tom Sample 10.wav,./Audio_Dataset/toms/Tom Sample 10.wav,469.67,1564.07,-526.057495,15.125816,14.260932,26.69183,10.05998,9.021091,...,-5.621459,-5.769226,-6.182073,28.643034,14.494631,17.842604,13.279398,16.241395,20.199276,40.587858
156,Tom Sample 12.wav,./Audio_Dataset/toms/Tom Sample 12.wav,429.62,1657.78,-468.365295,19.179003,16.861958,25.36055,11.88114,11.819507,...,-2.88387,-4.72994,-5.334169,27.207073,15.485105,17.586823,14.44896,16.067335,17.673515,38.67387
157,Tom Sample 5.wav,./Audio_Dataset/toms/Tom Sample 5.wav,758.22,2757.49,-482.770966,49.344799,27.762583,30.165825,16.573769,15.817742,...,-2.759051,-4.875149,-5.931608,24.590967,18.163811,15.16504,12.905531,13.673801,13.816301,43.832071
158,Tom Sample 4.wav,./Audio_Dataset/toms/Tom Sample 4.wav,691.61,2551.36,-491.714722,45.110367,25.816858,25.493032,12.545934,11.9184,...,-5.306271,-6.182037,-6.314246,23.852305,21.421249,15.184787,13.346305,14.775985,13.746772,43.269668


---
# **SAVE FEATURE FILES**


In [10]:
path = os.path.join(FEAT_FOLDER)
print("save features in " + path)

isExist = os.path.exists(path)
if not isExist:

   # Create a new directory because it does not exist
   os.makedirs(path)
   print("The new directory is created!")

print("save features to -> "+path + "feature.pkl")
with open(os.path.join(path, "feature.pkl"), 'wb') as fout:
   #pickle.dump(feat_dict, fout, protocol=pickle.HIGHEST_PROTOCOL)
   pickle.dump(feat_df, fout, protocol=pickle.HIGHEST_PROTOCOL)

save features in ./Features
save features to -> ./Featuresfeature.pkl
